In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [9]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [10]:
answer = assistant.rag(q['question'])

In [11]:
assistant.total_cost()

0.001164

In [12]:
print(answer)

You can join via **YouTube Live** even without the Zoom link. The Zoom link is only shared with instructors/presenters/TAs.

Check the **announcements channel on Telegram or Slack** for the video URL before it starts, or watch on the **DataTalksClub YouTube Channel**. Questions should be submitted via **Slido**, not in the chat.


In [13]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [14]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You can join via **YouTube Live** even without the Zoom link. The Zoom link is only shared with instructors/presenters/TAs.\n\nCheck the **announcements channel on Telegram or Slack** for the video URL before it starts, or watch on the **DataTalksClub YouTube Channel**. Questions should be submitted via **Slido**, not in the chat.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [15]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [16]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You can join the Office Hours or live workshop **without the Zoom link** by watching on **YouTube Live**.\n\n- The **Zoom link is only for instructors/presenters/TAs**.\n- For students, the **video URL is posted in the announcements channel on Telegram and Slack** before the session starts.\n- You can also watch live on the **DataTalksClub YouTube Channel**.\n- When live, submit questions in **Slido** — the link is pinned in the chat.\n\nAlso, **don’t post questions in chat**, since they may be missed if the room is busy.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksC

In [17]:
assistant.total_cost()

0.0025395

In [18]:
assistant.reset_usage()

In [19]:
assistant.total_cost()

0.0

In [20]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Filter the ground truth to documents we have.

In [28]:
ground_truth = [rec for rec in ground_truth if rec["document"] in doc_idx]

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [30]:
results[:10]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join late. If you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes — you can still join the course even if you missed the start date. You can start whenever you want, and the materials are available. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',


In [31]:
df_results = pd.DataFrame(results)

In [32]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join late. If you want a ce...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,Yes — you can still join the course even if yo...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,Yes — you can still join after the course has ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"Yes — you can still join, but to earn the cert...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [33]:
assistant.total_cost()


1.5830085000000003

In [34]:
df_results.to_csv("../data/rag-answers-new.csv", index=False)